# Irrigation Need — Notebook 1: EDA
**Kaggle Playground S6E4**  
Target: `Irrigation_Need` (Low / Medium / High)  
Metric: Balanced Accuracy  

Key prior: a community member reverse-engineered the exact generative formula from the original dataset.  
The formula uses only 6 raw features. Everything else is noise (or correlated filler).  
EDA will verify this and quantify how much the synthetic data drifted from the original.


## 0. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 60)
pd.set_option('display.float_format', '{:.3f}'.format)

PALETTE = {'Low': '#02C39A', 'Medium': '#F4A261', 'High': '#E63946'}
SEED = 42

# ── Formula feature names (the 6 raw columns that matter) ──
FORMULA_RAW = ['Soil_Moisture', 'Temperature_C', 'Rainfall_mm',
               'Wind_Speed_kmh', 'Crop_Growth_Stage', 'Mulching_Used']
FORMULA_THRESHOLDS = {
    'Soil_Moisture':  ('<', 25),
    'Temperature_C':  ('>', 30),
    'Rainfall_mm':    ('<', 300),
    'Wind_Speed_kmh': ('>', 10),
}

## 1. Load Data

In [ ]:
train = pd.read_csv('train.csv')
test  = pd.read_csv('test.csv')

print('=== SHAPES ===')
print(f'Train: {train.shape}')
print(f'Test:  {test.shape}')

print('\n=== COLUMNS ===')
print(train.columns.tolist())

print('\n=== DTYPES ===')
print(train.dtypes)

In [ ]:
train.head(10)

## 2. Target Distribution

In [ ]:
vc = train['Irrigation_Need'].value_counts()
vc_pct = train['Irrigation_Need'].value_counts(normalize=True) * 100

print('=== TARGET COUNTS ===')
for cls in ['Low', 'Medium', 'High']:
    print(f'  {cls:8s}: {vc.get(cls, 0):7,}  ({vc_pct.get(cls, 0):.1f}%)')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

order = ['Low', 'Medium', 'High']
colors = [PALETTE[c] for c in order]

axes[0].bar(order, [vc.get(c, 0) for c in order], color=colors, edgecolor='white', linewidth=0.5)
axes[0].set_title('Target class counts', fontweight='bold')
axes[0].set_ylabel('Count')
for i, cls in enumerate(order):
    axes[0].text(i, vc.get(cls, 0) + 100, f'{vc_pct.get(cls, 0):.1f}%', ha='center', fontsize=11)

axes[1].pie([vc.get(c, 0) for c in order], labels=order, colors=colors,
            autopct='%1.1f%%', startangle=90, textprops={'fontsize': 11})
axes[1].set_title('Target class proportions', fontweight='bold')

plt.suptitle('Irrigation_Need — Target Distribution', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

imbalance_ratio = vc.max() / vc.min()
print(f'\nImbalance ratio (max/min): {imbalance_ratio:.2f}')
print('NOTE: Metric is Balanced Accuracy — class imbalance matters.')

## 3. Missing Values

In [ ]:
def missing_report(df, label):
    miss = df.isnull().sum()
    miss = miss[miss > 0].sort_values(ascending=False)
    if len(miss) == 0:
        print(f'{label}: No missing values ✓')
    else:
        pct = (miss / len(df) * 100).round(2)
        print(f'{label} — {len(miss)} columns with missing values:')
        print(pd.DataFrame({'count': miss, 'pct': pct}).to_string())
    return miss

miss_train = missing_report(train, 'TRAIN')
print()
miss_test  = missing_report(test,  'TEST')

## 4. Categorical Features

In [ ]:
cat_cols = train.select_dtypes(include='object').columns.drop('Irrigation_Need', errors='ignore').tolist()
print(f'Categorical columns: {cat_cols}')

fig, axes = plt.subplots(1, len(cat_cols), figsize=(5 * len(cat_cols), 4))
if len(cat_cols) == 1:
    axes = [axes]

for ax, col in zip(axes, cat_cols):
    vc_cat = train[col].value_counts()
    ax.bar(vc_cat.index, vc_cat.values, color='#028090', edgecolor='white')
    ax.set_title(col, fontweight='bold')
    ax.set_ylabel('Count')
    ax.tick_params(axis='x', rotation=30)
    for i, (lbl, val) in enumerate(vc_cat.items()):
        ax.text(i, val + 50, f'{val:,}', ha='center', fontsize=9)

plt.suptitle('Categorical Feature Distributions', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Cross-tabs with target
for col in cat_cols:
    print(f'\n--- {col} × Irrigation_Need ---')
    ct = pd.crosstab(train[col], train['Irrigation_Need'], normalize='index') * 100
    print(ct.round(1).to_string())

## 5. Numeric Feature Distributions

In [ ]:
num_cols = train.select_dtypes(include=np.number).columns.drop('id').tolist()
print(f'Numeric columns ({len(num_cols)}): {num_cols}')

print('\n=== DESCRIPTIVE STATS (TRAIN) ===')
train[num_cols].describe().T

In [ ]:
# Distributions split by target class
n_cols = 4
n_rows = (len(num_cols) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 4 * n_rows))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    ax = axes[i]
    for cls in ['Low', 'Medium', 'High']:
        subset = train[train['Irrigation_Need'] == cls][col].dropna()
        ax.hist(subset, bins=40, alpha=0.5, label=cls, color=PALETTE[cls], density=True)
    ax.set_title(col, fontweight='bold', fontsize=10)
    ax.set_ylabel('Density')
    # Mark formula thresholds
    if col in FORMULA_THRESHOLDS:
        op, val = FORMULA_THRESHOLDS[col]
        ax.axvline(val, color='black', linestyle='--', linewidth=1.5, label=f'threshold={val}')
    if i == 0:
        ax.legend(fontsize=8)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Numeric Feature Distributions by Class\n(dashed line = formula threshold)', 
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 6. Formula Feature Analysis — The 6 That Matter

In [ ]:
# Compute binary formula indicators
def compute_formula_features(df):
    d = df.copy()
    d['soil_lt_25']  = (d['Soil_Moisture'] < 25).astype(int)
    d['temp_gt_30']  = (d['Temperature_C'] > 30).astype(int)
    d['rain_lt_300'] = (d['Rainfall_mm'] < 300).astype(int)
    d['wind_gt_10']  = (d['Wind_Speed_kmh'] > 10).astype(int)

    for stage in ['Flowering', 'Harvest', 'Sowing', 'Vegetative']:
        d[f'CGS_{stage}'] = (d['Crop_Growth_Stage'] == stage).astype(int)

    d['mulch_no']  = (d['Mulching_Used'] == 'No').astype(int)
    d['mulch_yes'] = (d['Mulching_Used'] == 'Yes').astype(int)

    # Logit scores
    d['logit_low'] = (
        16.3173 + (-11.0237*d['soil_lt_25']) + (-5.8559*d['temp_gt_30'])
        + (-10.8500*d['rain_lt_300']) + (-5.8284*d['wind_gt_10'])
        + (-5.4155*d['CGS_Flowering']) + (5.5073*d['CGS_Harvest'])
        + (5.2299*d['CGS_Sowing']) + (-5.4617*d['CGS_Vegetative'])
        + (-3.0014*d['mulch_no']) + (2.8613*d['mulch_yes'])
    )
    d['logit_medium'] = (
        4.6524 + (0.3290*d['soil_lt_25']) + (-0.0204*d['temp_gt_30'])
        + (0.1542*d['rain_lt_300']) + (0.0841*d['wind_gt_10'])
        + (0.3586*d['CGS_Flowering']) + (-0.1348*d['CGS_Harvest'])
        + (-0.3547*d['CGS_Sowing']) + (0.3334*d['CGS_Vegetative'])
        + (0.1883*d['mulch_no']) + (0.0142*d['mulch_yes'])
    )
    d['logit_high'] = (
        -20.9697 + (10.6947*d['soil_lt_25']) + (5.8763*d['temp_gt_30'])
        + (10.6958*d['rain_lt_300']) + (5.7444*d['wind_gt_10'])
        + (5.0569*d['CGS_Flowering']) + (-5.3725*d['CGS_Harvest'])
        + (-4.8752*d['CGS_Sowing']) + (5.1283*d['CGS_Vegetative'])
        + (2.8131*d['mulch_no']) + (-2.8755*d['mulch_yes'])
    )

    # Softmax probs
    logits = d[['logit_low', 'logit_medium', 'logit_high']].values
    exp_l  = np.exp(logits - logits.max(axis=1, keepdims=True))
    probs  = exp_l / exp_l.sum(axis=1, keepdims=True)
    d['prob_low']    = probs[:, 0]
    d['prob_medium'] = probs[:, 1]
    d['prob_high']   = probs[:, 2]
    d['formula_pred'] = ['Low','Medium','High'][np.argmax(probs, axis=1)]
    # vectorized version:
    cls_map = np.array(['Low','Medium','High'])
    d['formula_pred'] = cls_map[np.argmax(probs, axis=1)]

    # Simple integer score
    high_score = 2*d['soil_lt_25'] + 2*d['rain_lt_300'] + d['temp_gt_30'] + d['wind_gt_10']
    low_score  = 2*d['CGS_Harvest'] + 2*d['CGS_Sowing'] + d['mulch_yes']
    d['rule_score'] = high_score - low_score

    return d

train = compute_formula_features(train)
test  = compute_formula_features(test)
print('Formula features added.')

In [ ]:
# Formula accuracy on train
from sklearn.metrics import balanced_accuracy_score, confusion_matrix, classification_report

mask = train['Irrigation_Need'].notna() & train['formula_pred'].notna()
y_true = train.loc[mask, 'Irrigation_Need']
y_pred = train.loc[mask, 'formula_pred']

ba = balanced_accuracy_score(y_true, y_pred)
print(f'Formula Balanced Accuracy on synthetic train: {ba:.4f}')
print(f'(1.0 on original 10k rows — gap shows synthetic noise level)\n')
print(classification_report(y_true, y_pred))

cm = confusion_matrix(y_true, y_pred, labels=['Low', 'Medium', 'High'])
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Low','Medium','High'],
            yticklabels=['Low','Medium','High'], ax=ax)
ax.set_xlabel('Predicted', fontweight='bold')
ax.set_ylabel('Actual', fontweight='bold')
ax.set_title(f'Formula Confusion Matrix (BA={ba:.4f})', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Where does the formula fail? Characterise the residuals
train['formula_correct'] = (train['formula_pred'] == train['Irrigation_Need'])
wrong = train[~train['formula_correct']]
right = train[train['formula_correct']]

print(f'Formula wrong on {len(wrong):,} rows ({len(wrong)/len(train)*100:.1f}%)')
print(f'\nError breakdown by true class:')
print(wrong['Irrigation_Need'].value_counts())
print(f'\nError breakdown by predicted class:')
print(wrong['formula_pred'].value_counts())

# Distribution of rule_score for correct vs wrong
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for subset, label, color, ax in [
    (right, 'Correct', '#02C39A', axes[0]),
    (wrong, 'Wrong',   '#E63946', axes[1])
]:
    ax.hist(subset['rule_score'], bins=range(-5, 8), color=color, alpha=0.8, edgecolor='white')
    ax.axvline(0, color='black', linestyle='--', linewidth=1)
    ax.axvline(3, color='black', linestyle='--', linewidth=1)
    ax.set_title(f'Rule score — {label} predictions', fontweight='bold')
    ax.set_xlabel('Rule score')
    ax.set_ylabel('Count')

plt.suptitle('Rule score distribution: formula correct vs wrong\n(boundaries at 0 and 3)', 
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Train vs Test Distribution Shift

In [ ]:
# KS test: are train and test drawn from the same distribution per feature?
from scipy.stats import ks_2samp

ks_results = []
for col in num_cols:
    stat, pval = ks_2samp(train[col].dropna(), test[col].dropna())
    ks_results.append({'feature': col, 'ks_stat': stat, 'p_value': pval,
                       'shift': 'YES' if pval < 0.05 else 'no'})

ks_df = pd.DataFrame(ks_results).sort_values('ks_stat', ascending=False)
print('=== KS Test: Train vs Test Distribution Shift ===')
print(ks_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#E63946' if s == 'YES' else '#02C39A' for s in ks_df['shift']]
ax.barh(ks_df['feature'], ks_df['ks_stat'], color=colors)
ax.axvline(0.05, color='black', linestyle='--', linewidth=1, label='p=0.05 approx')
ax.set_xlabel('KS Statistic (higher = more shift)')
ax.set_title('Train vs Test distribution shift by feature\n(red = statistically significant at p<0.05)', 
             fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

## 8. Correlation & Redundancy Among Numeric Features

In [ ]:
corr = train[num_cols].corr()

fig, ax = plt.subplots(figsize=(14, 11))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            vmin=-1, vmax=1, linewidths=0.4, ax=ax, annot_kws={'size': 7})
ax.set_title('Numeric Feature Correlation Matrix (lower triangle)', fontweight='bold')
plt.tight_layout()
plt.show()

# Pairs with |r| > 0.6
high_corr = []
for i in range(len(corr.columns)):
    for j in range(i+1, len(corr.columns)):
        r = corr.iloc[i, j]
        if abs(r) > 0.6:
            high_corr.append({'feat_A': corr.columns[i], 'feat_B': corr.columns[j], 'r': round(r, 3)})

if high_corr:
    print('Highly correlated pairs (|r| > 0.6):')
    print(pd.DataFrame(high_corr).sort_values('r', key=abs, ascending=False).to_string(index=False))
else:
    print('No pairs with |r| > 0.6 — no obvious collinear redundancy.')

## 9. Formula Feature Separability

In [ ]:
# How cleanly do the formula thresholds separate classes?
threshold_features = {
    'Soil_Moisture':  25,
    'Temperature_C':  30,
    'Rainfall_mm':    300,
    'Wind_Speed_kmh': 10,
}

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
axes = axes.flatten()

for ax, (col, thresh) in zip(axes, threshold_features.items()):
    for cls in ['Low', 'Medium', 'High']:
        subset = train[train['Irrigation_Need'] == cls][col].dropna()
        ax.hist(subset, bins=60, alpha=0.55, label=cls, color=PALETTE[cls], density=True)
    ax.axvline(thresh, color='black', linestyle='--', linewidth=2,
               label=f'threshold = {thresh}')
    ax.set_title(col, fontweight='bold')
    ax.set_xlabel(col)
    ax.set_ylabel('Density')
    ax.legend(fontsize=8)

plt.suptitle('Formula Threshold Features — Class Overlap', 
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

# What % of each class is on the 'formula-expected' side?
print('--- % on formula-expected side of threshold ---')
for col, thresh in threshold_features.items():
    op = '<' if col in ['Soil_Moisture', 'Rainfall_mm'] else '>'
    print(f'\n{col} {op} {thresh}:')
    for cls in ['Low', 'Medium', 'High']:
        sub = train[train['Irrigation_Need'] == cls][col]
        if op == '<':
            pct = (sub < thresh).mean() * 100
        else:
            pct = (sub > thresh).mean() * 100
        print(f'  {cls:8s}: {pct:.1f}%')

In [ ]:
# Rule score vs true class — key diagnostic
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Boxplot of rule_score by class
order = ['Low', 'Medium', 'High']
data_box = [train[train['Irrigation_Need'] == cls]['rule_score'] for cls in order]
bp = axes[0].boxplot(data_box, labels=order, patch_artist=True,
                     boxprops=dict(linewidth=1.5),
                     medianprops=dict(color='black', linewidth=2))
for patch, cls in zip(bp['boxes'], order):
    patch.set_facecolor(PALETTE[cls])
    patch.set_alpha(0.7)
axes[0].axhline(0, color='grey', linestyle='--', linewidth=1)
axes[0].axhline(3, color='grey', linestyle='--', linewidth=1)
axes[0].set_title('Rule score by true class\n(boundaries: 0 and 3)', fontweight='bold')
axes[0].set_ylabel('Rule score')

# Stacked bar: rule_score value → class distribution
score_vals = sorted(train['rule_score'].unique())
bottom = np.zeros(len(score_vals))
for cls in order:
    counts = [train[(train['rule_score'] == s) & (train['Irrigation_Need'] == cls)].shape[0]
              for s in score_vals]
    axes[1].bar(score_vals, counts, bottom=bottom, label=cls, color=PALETTE[cls], alpha=0.85)
    bottom += np.array(counts)
axes[1].axvline(0.5, color='black', linestyle='--', linewidth=1.5)
axes[1].axvline(3.5, color='black', linestyle='--', linewidth=1.5)
axes[1].set_title('Class composition per rule score value', fontweight='bold')
axes[1].set_xlabel('Rule score')
axes[1].set_ylabel('Count')
axes[1].legend()

plt.suptitle('Rule Score Separability', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 10. Non-Formula Features — Do Any Add Signal?

In [ ]:
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import LabelEncoder

non_formula_num = [c for c in num_cols if c not in FORMULA_RAW + list(threshold_features.keys())]
print(f'Non-formula numeric features: {non_formula_num}')

le_target = LabelEncoder()
y_enc = le_target.fit_transform(train['Irrigation_Need'])

# Mutual information
X_nf = train[non_formula_num].fillna(train[non_formula_num].median())
mi_scores = mutual_info_classif(X_nf, y_enc, random_state=SEED)
mi_df = pd.DataFrame({'feature': non_formula_num, 'MI': mi_scores}).sort_values('MI', ascending=False)

print('\nMutual Information with target (non-formula features):')
print(mi_df.to_string(index=False))

# Also MI for formula features for comparison
formula_cols_present = [c for c in FORMULA_RAW if c in num_cols]
X_f = train[formula_cols_present].fillna(train[formula_cols_present].median())
mi_f = mutual_info_classif(X_f, y_enc, random_state=SEED)
mi_f_df = pd.DataFrame({'feature': formula_cols_present, 'MI': mi_f}).sort_values('MI', ascending=False)
print('\nMutual Information — formula numeric features (for reference):')
print(mi_f_df.to_string(index=False))

In [ ]:
# Visual: MI comparison all features
mi_all = pd.concat([mi_f_df, mi_df]).sort_values('MI', ascending=True)
colors_mi = ['#028090' if f in FORMULA_RAW else '#ccc' for f in mi_all['feature']]

fig, ax = plt.subplots(figsize=(9, max(5, len(mi_all)*0.4)))
ax.barh(mi_all['feature'], mi_all['MI'], color=colors_mi)
ax.set_xlabel('Mutual Information with Irrigation_Need')
ax.set_title('Feature MI with target\n(teal = formula features)', fontweight='bold')
plt.tight_layout()
plt.show()

## 11. Outlier Detection

In [ ]:
# IQR-based outlier counts per column
outlier_report = []
for col in num_cols:
    q1, q3 = train[col].quantile(0.25), train[col].quantile(0.75)
    iqr = q3 - q1
    lo, hi = q1 - 1.5*iqr, q3 + 1.5*iqr
    n_out = ((train[col] < lo) | (train[col] > hi)).sum()
    outlier_report.append({'feature': col, 'Q1': q1, 'Q3': q3,
                           'lower_fence': lo, 'upper_fence': hi,
                           'n_outliers': n_out,
                           'pct': round(n_out / len(train) * 100, 2)})

out_df = pd.DataFrame(outlier_report).sort_values('n_outliers', ascending=False)
print('=== IQR Outlier Report ===')
print(out_df.to_string(index=False))

In [ ]:
# Boxplots for formula features to spot outliers / anomalies
formula_num = [c for c in FORMULA_THRESHOLDS.keys() if c in train.columns]

fig, axes = plt.subplots(1, len(formula_num), figsize=(14, 5))
for ax, col in zip(axes, formula_num):
    data_cls = [train[train['Irrigation_Need'] == cls][col] for cls in ['Low', 'Medium', 'High']]
    bp = ax.boxplot(data_cls, labels=['Low','Med','High'], patch_artist=True,
                    medianprops=dict(color='black', linewidth=2))
    for patch, cls in zip(bp['boxes'], ['Low','Medium','High']):
        patch.set_facecolor(PALETTE[cls])
        patch.set_alpha(0.7)
    thresh = FORMULA_THRESHOLDS[col][1]
    ax.axhline(thresh, color='black', linestyle='--', linewidth=1.5, label=f'threshold={thresh}')
    ax.set_title(col, fontweight='bold')
    ax.legend(fontsize=8)

plt.suptitle('Formula Features — Boxplots by Class with Threshold', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 12. Rainfall_mm Anomaly Check
From the Kaggle data explorer, `Rainfall_mm` showed a suspicious gap around 300–450mm and a cluster near 0. Worth examining closely — this is a formula-critical feature.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(train['Rainfall_mm'], bins=100, color='#028090', edgecolor='none', alpha=0.8)
axes[0].axvline(300, color='red', linestyle='--', linewidth=2, label='threshold=300')
axes[0].set_title('Rainfall_mm — full distribution', fontweight='bold')
axes[0].set_xlabel('Rainfall_mm')
axes[0].legend()

# Zoom in on low end
axes[1].hist(train[train['Rainfall_mm'] < 400]['Rainfall_mm'], 
             bins=80, color='#F4A261', edgecolor='none', alpha=0.8)
axes[1].axvline(300, color='red', linestyle='--', linewidth=2, label='threshold=300')
axes[1].set_title('Rainfall_mm — zoomed below 400mm', fontweight='bold')
axes[1].set_xlabel('Rainfall_mm')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Rows with Rainfall_mm < 10  : {(train['Rainfall_mm'] < 10).sum():,}")
print(f"Rows with Rainfall_mm < 50  : {(train['Rainfall_mm'] < 50).sum():,}")
print(f"Rows with Rainfall_mm < 300 : {(train['Rainfall_mm'] < 300).sum():,}")
print(f"Rows with Rainfall_mm >= 300: {(train['Rainfall_mm'] >= 300).sum():,}")
print(f"Rows with Rainfall_mm = 0   : {(train['Rainfall_mm'] == 0).sum():,}")

## 13. EDA Summary & Decisions for Notebook 2 (Cleaning)

In [ ]:
print('=' * 65)
print('EDA SUMMARY')
print('=' * 65)

print(f"""
DATASET
  Train rows : {len(train):,}
  Test rows  : {len(test):,}
  Features   : {train.shape[1]} (incl. formula-engineered)

TARGET
  Distribution: see cell 2 printout
  Imbalance ratio: {vc.max()/vc.min():.2f}x
  → Need class weighting or resampling in Notebook 2

MISSING VALUES
  {'None found ✓' if len(miss_train) == 0 else f'{len(miss_train)} columns — see cell 3'}

FORMULA PERFORMANCE
  Balanced accuracy on synthetic train: {ba:.4f}
  → Gap from 1.0 = synthetic noise. Model must capture this residual.
  → Most errors occur at boundary scores (0 and 3). 
     The boundary zone is where ML adds value.

KEY CLEANING DECISIONS FOR NOTEBOOK 2
  1. Drop 'id' column (index only)
  2. Flag and inspect extreme Rainfall_mm outliers (near 0)
  3. Encode Crop_Growth_Stage and Mulching_Used (label or OHE)
  4. Add all formula features as new columns
  5. Rebalance: use class_weight='balanced' in models,
     optionally oversample with SMOTE on non-formula noise features
  6. Consider dropping low-MI non-formula features
  7. Threshold tuning: rule_score boundaries 0 and 3 may shift
     slightly on synthetic data — tune in modelling notebooks

FEATURES TO DEFINITELY KEEP
  Soil_Moisture, Temperature_C, Rainfall_mm, Wind_Speed_kmh,
  Crop_Growth_Stage, Mulching_Used  +  all formula-derived cols

FEATURES TO EVALUATE (low MI expected)
  Soil_pH, Organic_Carbon, Electrical_Conductivity,
  Humidity, Sunlight_Hours, Soil_Type
  → Keep in tree models (free), drop for logistic/linear baselines
""")

print('=' * 65)
print('→ Proceed to 02_Cleaning_Rebalancing.ipynb')
print('=' * 65)